In [1]:
import os
import math
import heapq
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# 1. Load Road Network Data
df_nodes = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet"))
df_edges = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"))

node_dict = {row["node_id"]: row.to_dict() for _, row in df_nodes.iterrows()}
edge_dict = {row["edge_id"]: row.to_dict() for _, row in df_edges.iterrows()}

# Build adjacency list: u -> list of (v, edge_id)
adj_list = {row["node_id"]: [] for _, row in df_nodes.iterrows()}
for _, row in df_edges.iterrows():
    adj_list[row["u"]].append((row["v"], row["edge_id"]))

MAX_NETWORK_SPEED_MPS = 110.0 * (1000.0 / 3600.0)  # ~30.5 m/s

def haversine_m(lon1, lat1, lon2, lat2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    return 6371000.0 * 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

def haversine_heuristic(u, goal):
    u_node = node_dict[u]
    g_node = node_dict[goal]
    dist = haversine_m(u_node["lon"], u_node["lat"], g_node["lon"], g_node["lat"])
    return dist / MAX_NETWORK_SPEED_MPS

# Function to get time-dependent weight (interpolating or selecting by time horizon)
def get_edge_weight(edge_id, current_time_s, edge_penalties=None):
    e = edge_dict[edge_id]
    
    # Map current time in seconds to closest precomputed horizon (0h, 3h, 6h, 12h)
    t_hr = current_time_s / 3600.0
    if t_hr <= 1.5:
        base_w = e["weight_t0_s"]
    elif t_hr <= 4.5:
        base_w = e["weight_t3_s"]
    elif t_hr <= 9.0:
        base_w = e["weight_t6_s"]
    else:
        base_w = e["weight_t12_s"]
        
    penalty = edge_penalties.get(edge_id, 1.0) if edge_penalties else 1.0
    return base_w * penalty

# 2. Time-Dependent A* Search Implementation
def time_dependent_astar(start_node, goal_node, departure_time_s=0.0, edge_penalties=None):
    """
    Finds the least-cost dynamic path from start to goal avoiding closed or high-hazard edges.
    Returns: (path_nodes, path_edges, total_travel_time_s, cumulative_burn_risk)
    """
    # Priority queue stores (f_score, g_score, current_node, path_nodes, path_edges)
    pq = []
    h0 = haversine_heuristic(start_node, goal_node)
    heapq.heappush(pq, (h0, 0.0, start_node, [start_node], []))
    
    visited = {}
    
    while pq:
        f, g, u, path_nodes, path_edges = heapq.heappop(pq)
        
        if u == goal_node:
            # Calculate cumulative burn risk along chosen corridor
            burn_risk = sum(edge_dict[eid]["burn_prob_t3"] for eid in path_edges)
            return path_nodes, path_edges, g, burn_risk
            
        if u in visited and visited[u] <= g:
            continue
        visited[u] = g
        
        for v, eid in adj_list[u]:
            w = get_edge_weight(eid, departure_time_s + g, edge_penalties)
            if w >= 1e8:  # Impassable / Closed edge
                continue
                
            new_g = g + w
            new_f = new_g + haversine_heuristic(v, goal_node)
            heapq.heappush(pq, (new_f, new_g, v, path_nodes + [v], path_edges + [eid]))
            
    return None, None, float("inf"), float("inf")

# 3. Disjoint Evacuation Corridor Generator (Primary + Secondary Path)
def find_disjoint_corridors(start_node, goal_node, departure_time_s=0.0, max_overlap_frac=0.10):
    # Primary Route
    p_nodes, p_edges, p_time, p_risk = time_dependent_astar(start_node, goal_node, departure_time_s)
    if not p_edges:
        return None, None
        
    # Penalize primary edges heavily (e.g. 50x) to force path diversity
    penalties = {eid: 50.0 for eid in p_edges}
    
    # Secondary Route
    s_nodes, s_edges, s_time, s_risk = time_dependent_astar(start_node, goal_node, departure_time_s, edge_penalties=penalties)
    
    if s_edges:
        # Measure edge overlap
        overlap_edges = set(p_edges).intersection(set(s_edges))
        overlap_frac = len(overlap_edges) / max(1, len(p_edges))
    else:
        s_nodes, s_edges, s_time, s_risk = [], [], float("inf"), float("inf")
        overlap_frac = 0.0
        
    primary_route = {
        "nodes": p_nodes,
        "edges": p_edges,
        "travel_time_s": p_time,
        "burn_risk": p_risk
    }
    
    secondary_route = {
        "nodes": s_nodes,
        "edges": s_edges,
        "travel_time_s": s_time,
        "burn_risk": s_risk,
        "overlap_fraction": overlap_frac
    }
    
    return primary_route, secondary_route

# 4. Test Evacuation Query: Paradise Center (1001) -> Chico Downtown (1004) at T=0h
prim_0h, sec_0h = find_disjoint_corridors(1001, 1004, departure_time_s=0.0)

print("--- Evacuation Routing Query (Paradise -> Chico Downtown @ T=0h) ---")
print(f"Primary Corridor:   Nodes: {prim_0h['nodes']}")
print(f"                    Travel Time: {prim_0h['travel_time_s']/60:.1f} min | Risk: {prim_0h['burn_risk']:.4f}")
if sec_0h['edges']:
    print(f"Secondary Corridor: Nodes: {sec_0h['nodes']}")
    print(f"                    Travel Time: {sec_0h['travel_time_s']/60:.1f} min | Risk: {sec_0h['burn_risk']:.4f} | Overlap: {sec_0h['overlap_fraction']*100:.1f} %")
else:
    print("Secondary Corridor: No independent route found.")

--- Evacuation Routing Query (Paradise -> Chico Downtown @ T=0h) ---
Primary Corridor:   Nodes: [1001, 1009, 1011, 1003, 1004]
                    Travel Time: 24.1 min | Risk: 2.9128
Secondary Corridor: Nodes: [1001, 1002, 1007, 1014, 1003, 1004]
                    Travel Time: 336.5 min | Risk: 4.4139 | Overlap: 25.0 %


In [2]:
import os
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Define Evacuation Origins (Population centers) and Safe Hubs
evacuation_origins = [
    {"node_id": 1001, "name": "Paradise_Center", "evacuees": 4800},
    {"node_id": 1002, "name": "Magalia_Upper", "evacuees": 2600},
    {"node_id": 1008, "name": "Concow_Penta_Jct", "evacuees": 1385}
]

safe_destinations = [
    {"node_id": 1004, "name": "Chico_Downtown"},
    {"node_id": 1006, "name": "Oroville_Center"}
]

# 2. Compute Multi-Origin Disjoint Corridors
all_evacuation_routes = []

print("--- Computing Multi-Origin Evacuation Plans ---")
for orig in evacuation_origins:
    o_id = orig["node_id"]
    o_name = orig["name"]
    
    # Evaluate routes to both safe destinations and choose the optimal pair
    best_dest = None
    best_prim, best_sec = None, None
    min_prim_time = float("inf")
    
    for dest in safe_destinations:
        d_id = dest["node_id"]
        p_route, s_route = find_disjoint_corridors(o_id, d_id, departure_time_s=0.0)
        
        if p_route and p_route["travel_time_s"] < min_prim_time:
            min_prim_time = p_route["travel_time_s"]
            best_dest = dest
            best_prim = p_route
            best_sec = s_route
            
    plan_entry = {
        "origin_id": o_id,
        "origin_name": o_name,
        "destination_id": best_dest["node_id"],
        "destination_name": best_dest["name"],
        "evacuees": orig["evacuees"],
        "primary_corridor": best_prim,
        "secondary_corridor": best_sec
    }
    all_evacuation_routes.append(plan_entry)
    
    print(f"Origin: {o_name:<18} -> Safe Hub: {best_dest['name']:<16}")
    print(f"  Primary Path:   {best_prim['nodes']} | Time: {best_prim['travel_time_s']/60:.1f} min")
    print(f"  Secondary Path: {best_sec['nodes']} | Time: {best_sec['travel_time_s']/60:.1f} min | Overlap: {best_sec['overlap_fraction']*100:.1f} %")

# 3. Mesoscopic Clearance Time Simulation (Dynamic Flow Loading)
# Average vehicle occupancy = 2.0 persons per vehicle
# Release rate = 800 vehicles per hour per origin corridor
simulation_records = []
VEHICLE_OCCUPANCY = 2.0

for plan in all_evacuation_routes:
    total_vehicles = plan["evacuees"] / VEHICLE_OCCUPANCY
    
    # 70% of evacuees take primary route, 30% take secondary/alternate route
    prim_vehs = total_vehicles * 0.70
    sec_vehs = total_vehicles * 0.30
    
    # Primary corridor bottleneck capacity
    prim_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["primary_corridor"]["edges"]]
    prim_bottleneck_cap = min(prim_edge_caps)
    
    # Clearance time on primary corridor: Departure spreading time + traversal time
    prim_clearance_hr = (prim_vehs / prim_bottleneck_cap) + (plan["primary_corridor"]["travel_time_s"] / 3600.0)
    
    # Secondary corridor clearance
    if plan["secondary_corridor"]["edges"]:
        sec_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["secondary_corridor"]["edges"]]
        sec_bottleneck_cap = min(sec_edge_caps)
        sec_clearance_hr = (sec_vehs / sec_bottleneck_cap) + (plan["secondary_corridor"]["travel_time_s"] / 3600.0)
    else:
        sec_clearance_hr = prim_clearance_hr * 1.5
        
    overall_95_clearance_hr = max(prim_clearance_hr, sec_clearance_hr)
    
    simulation_records.append({
        "origin_name": plan["origin_name"],
        "destination_name": plan["destination_name"],
        "total_evacuees": plan["evacuees"],
        "total_vehicles": int(total_vehicles),
        "primary_bottleneck_cap_vph": prim_bottleneck_cap,
        "primary_freeflow_min": np.float32(plan["primary_corridor"]["travel_time_s"] / 60.0),
        "primary_clearance_hr": np.float32(prim_clearance_hr),
        "secondary_clearance_hr": np.float32(sec_clearance_hr),
        "clearance_time_95_hr": np.float32(overall_95_clearance_hr)
    })

df_sim = pd.DataFrame(simulation_records)

print("\n=======================================================")
print("MESOSCOPIC EVACUATION CLEARANCE TIME RESULTS")
print("=======================================================")
print(df_sim[["origin_name", "total_evacuees", "total_vehicles", "primary_clearance_hr", "clearance_time_95_hr"]])
print("=======================================================")
print(f"Full Regional 95% Population Clearance Time: {df_sim['clearance_time_95_hr'].max():.2f} hours")

# 4. Save JSON and Parquet Outputs
json_routes_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_routes.json")
with open(json_routes_path, "w") as f:
    json.dump(all_evacuation_routes, f, indent=2)

sim_parquet_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_simulation_results.parquet")
df_sim.to_parquet(sim_parquet_path, compression="snappy")

print(f"\nSaved artifacts:")
print(f" - {json_routes_path}")
print(f" - {sim_parquet_path}")

--- Computing Multi-Origin Evacuation Plans ---
Origin: Paradise_Center    -> Safe Hub: Oroville_Center 
  Primary Path:   [1001, 1010, 1012, 1005, 1006] | Time: 23.7 min
  Secondary Path: [1001, 1009, 1011, 1005, 1006] | Time: 184.4 min | Overlap: 25.0 %
Origin: Magalia_Upper      -> Safe Hub: Oroville_Center 
  Primary Path:   [1002, 1001, 1010, 1012, 1005, 1006] | Time: 32.4 min
  Secondary Path: [1002, 1007, 1014, 1003, 1011, 1005, 1006] | Time: 217.0 min | Overlap: 20.0 %
Origin: Concow_Penta_Jct   -> Safe Hub: Oroville_Center 
  Primary Path:   [1008, 1012, 1005, 1006] | Time: 25.3 min
  Secondary Path: [1008, 1001, 1009, 1011, 1005, 1006] | Time: 194.7 min | Overlap: 33.3 %

MESOSCOPIC EVACUATION CLEARANCE TIME RESULTS
        origin_name  total_evacuees  total_vehicles  primary_clearance_hr  \
0   Paradise_Center            4800            2400              0.836744   
1     Magalia_Upper            2600            1300              0.954198   
2  Concow_Penta_Jct            13